In [87]:
# ID

prenom = "Chloé"
nom = "Makoundou"
formation = "M1 IBD"

print(f"Prénom : {prenom}\nNom : {nom}\nFormation : {formation}")

Prénom : Chloé
Nom : Makoundou
Formation : M1 IBD


# **TP 2 — Préparation des données avec Python**

## Objectif
Nettoyer le dataset `catnat_dirty.csv` et produire un fichier propre `catnat_clean.csv` prêt à être analysé dans Tableau.

### Problèmes à résoudre
Le dataset contient volontairement :

- ~150 doublons
- Valeurs manquantes supplémentaires
- Incohérences de casse (Asia, ASIA, asia...)
- Espaces parasites
- Variantes d'orthographe (USA, US, United States...)
- `Start Year` en format texte avec erreurs ("2020 AD", "Year 2020")
- Outliers aberrants (décès négatifs, magnitude à 999)

In [2]:
# Installation library

In [3]:
# Importation des bibliothèques
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("datas/catnat_dirty.csv")

## Exercice 1 — Exploration et diagnostic

In [5]:
# 1. Afficher les dimensions du dataset (nombre de lignes et de colonnes).
df.shape

(17510, 18)

In [6]:
# 2. Affichez les types de données avec info()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17510 entries, 0 to 17509
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   DisNo.                   17510 non-null  object 
 1   Country                  17510 non-null  object 
 2   Region                   17161 non-null  object 
 3   Subregion                17510 non-null  object 
 4   Disaster Type            17510 non-null  object 
 5   Disaster Subtype         17510 non-null  object 
 6   Disaster Subgroup        17510 non-null  object 
 7   Event Name               3975 non-null   object 
 8   Start Year               17510 non-null  object 
 9   Start Month              16599 non-null  float64
 10  Total Deaths             12593 non-null  float64
 11  No. Injured              4507 non-null   float64
 12  Total Affected           12951 non-null  float64
 13  No. Homeless             2521 non-null   float64
 14  Total Damage ('000 US$

In [7]:
# 3. Comptez les valeurs manquantes par colonne (nombre et pourcentage)
valeur_manquante = df.isnull().sum()
print("Nombre de valeurs manquantes par colonne:\n", valeur_manquante)

Nombre de valeurs manquantes par colonne:
 DisNo.                         0
Country                        0
Region                       349
Subregion                      0
Disaster Type                  0
Disaster Subtype               0
Disaster Subgroup              0
Event Name                 13535
Start Year                     0
Start Month                  911
Total Deaths                4917
No. Injured                13003
Total Affected              4559
No. Homeless               14989
Total Damage ('000 US$)    11880
Magnitude                  12261
Latitude                   14689
Longitude                  14689
dtype: int64


In [8]:
#4. Comptez le nombre de doublons
nb_doublons = df.duplicated().sum()
print(f"Nombre de doublons dans le dataset : {nb_doublons}")

Nombre de doublons dans le dataset : 150


In [9]:
# 5. Affichez les valeurs uniques de Region — repérez les incohérences
unique_val_region = df["Region"].value_counts()
print("Valeurs uniques de la colonne 'Region':\n", unique_val_region)

Valeurs uniques de la colonne 'Region':
 Region
Asia          3754
Americas      2359
Africa        1688
Europe        1201
ASIA           792
asia           763
americas       504
AMERICAS       486
Asia           429
 Asia          396
Oceania        396
AFRICA         375
africa         370
 Asia          282
EUROPE         265
 Americas      248
europe         232
Americas       231
 Africa        202
Africa         181
 Americas      178
 Africa        140
 ASIA          116
Europe         105
 Europe        100
OCEANIA         84
 asia           82
ASIA            77
oceania         75
 Europe         75
asia            69
 asia           61
 ASIA           60
 AMERICAS       57
americas        56
AMERICAS        46
 americas       44
 AMERICAS       42
AFRICA          42
Oceania         41
 AFRICA         38
 Oceania        38
 africa         37
africa          37
 AFRICA         34
 americas       32
 europe         31
EUROPE          31
europe          30
 EUROPE         26
 a

In [10]:
unique_val_region2 = df["Region"].unique()
print("Valeurs uniques de la colonne 'Region':\n", unique_val_region2)

Valeurs uniques de la colonne 'Region':
 ['Asia ' 'Americas' 'Oceania' 'Asia' 'ASIA' ' Americas ' 'Africa'
 'Europe ' 'Europe' ' Americas' 'EUROPE' 'AMERICAS ' 'americas' 'AMERICAS'
 nan ' Asia ' ' Africa' 'Americas ' 'asia' ' Oceania' 'ASIA ' 'oceania'
 ' americas ' ' Asia' 'asia ' 'OCEANIA' 'africa' 'AFRICA' 'europe'
 'europe ' ' AMERICAS ' 'americas ' ' AMERICAS' ' Europe ' ' EUROPE'
 ' ASIA' ' Africa ' ' americas' 'africa ' ' asia' ' africa' ' AFRICA '
 'Africa ' ' AFRICA' 'Oceania ' ' asia ' 'AFRICA ' ' europe ' 'EUROPE '
 ' Europe' ' africa ' ' ASIA ' ' europe' ' Oceania ' 'OCEANIA '
 ' OCEANIA ' ' EUROPE ' 'oceania ' ' OCEANIA' ' oceania ' ' oceania']


**incohérences observées :** 
- `Region` : "ASIA", "asia", "Asia ", "EUROPE", "EUROPE ", etc. Majuscules, minuscules, espaces
- `Country` : "USA", "US", "United States", "United States of America", etc. variantes d'orthographe
- `oceania` : "Oceania", "oceania ", "OCEANIA", etc. doublons
- `nan`


In [11]:
# 6. Affichez les statistiques de Total Deaths — repérez les anomalies
stats_total_deaths = df["Total Deaths"].describe()
stats_total_deaths

count    1.259300e+04
mean     2.751663e+03
std      6.577989e+04
min     -3.600000e+04
25%      5.000000e+00
50%      1.800000e+01
75%      6.100000e+01
max      3.700000e+06
Name: Total Deaths, dtype: float64

### Questions
- Combien y a-t-il de doublons ?

> 150
- Quelles colonnes ont le plus de valeurs manquantes ? (Top 3)

> 1) No. Homeless = 14989, 
> 2) Latitude = 14689,
> 3) Longitude = 14689
- Combien de variantes différentes pour "Asia" ?
> 12 variantes différentes pour Asia.
> 'Asia '
'Asia'
'ASIA'
' Asia '
'asia'
'ASIA '
' Asia'
'asia '
' ASIA'
' asia'
' asia '
' ASIA '

- Y a-t-il des valeurs négatives dans Total Deaths ?
> oui il y a des valeurs négatives dans Total Deaths. (min = -3.600000e+04)


## Exercice 2 — Suppression des doublons

In [12]:
# 1. Affichez quelques lignes dupliquées pour vérifier

df[df.duplicated()]
df[df.duplicated(keep=False)].head(10)

,DisNo.,Country,Region,Subregion,Disaster Type,Disaster Subtype,Disaster Subgroup,Event Name,Start Year,Start Month,Total Deaths,No. Injured,Total Affected,No. Homeless,Total Damage ('000 US$),Magnitude,Latitude,Longitude
27,1969-0071-IND,India,ASIA,Southern Asia,Storm,Tropical cyclone,Meteorological,NaN,1969,5.0,600000.0,NaN,260000.0,NaN,8330.0,NaN,NaN,NaN
125,2005-0583-USA,USA,americas,Northern America,Flood,Riverine flood,Hydrological,NaN,2005,10.0,11.0,NaN,3000.0,NaN,NaN,38290.0,NaN,NaN
322,1963-0055-BEL,Belgium,Europe,Western Europe,Extreme Temperature,Cold wave,Meteorological,NaN,1963,NaN,12.0,NaN,NaN,NaN,NaN,-22.0,NaN,NaN
371,2023-0510-MNG,Mongolia,Asia,Eastern Asia,Flood,Flash flood,Hydrological,NaN,2023,8.0,4.0,NaN,1230.0,NaN,NaN,NaN,NaN,NaN
456,1996-0226-CHN,China,Asia,Eastern Asia,Storm,Tropical cyclone,Meteorological,Willie,1996,9.0,38.0,NaN,NaN,NaN,100000.0,130.0,NaN,NaN
489,2025-0173-THA,Thailand,NaN,South-eastern Asia,Storm,Severe weather,Meteorological,NaN,2025,3.0,NaN,NaN,4231.0,NaN,NaN,NaN,NaN,NaN
494,2018-0436-AGO,Angola,Africa,Sub-Saharan Africa,epidemic,Bacterial disease,Biological,Cholera,2018,10.0,2.0,NaN,139.0,NaN,NaN,NaN,NaN,NaN
570,2021-0596-PAK,Pakistan,Asia,Southern Asia,Flood,Flood (General),Hydrological,NaN,2021,9.0,19000.0,4.0,4.0,NaN,NaN,NaN,NaN,NaN
586,1989-0224-CHN,China,ASIA,Eastern Asia,Storm,Tropical cyclone,Meteorological,Brian,1989,10.0,31.0,700.0,700.0,NaN,NaN,NaN,NaN,NaN
617,2002-0406-JPN,JAPAN,Asia,Eastern Asia,Storm,Storm (General),Meteorological,NaN,2002,7.0,1.0,NaN,NaN,NaN,500.0,NaN,NaN,NaN


In [13]:
# 2. Supprimez les doublons exacts

df = df.drop_duplicates()

In [14]:
# 3. Vérifiez que les doublons ont bien été supprimés

df.duplicated().sum()

np.int64(0)

Il n'y a plus de doublons 

## Exercice 3 — Correction des types

In [15]:
# 1. Verifiez les type de Start Year

df["Start Year"].dtype

dtype('O')

In [16]:
# 2. Affichez quelques valeurs problématiques (contenant "Year" ou "AD")

df[df["Start Year"].str.contains("Year|AD", na=False)].head(5)

,DisNo.,Country,Region,Subregion,Disaster Type,Disaster Subtype,Disaster Subgroup,Event Name,Start Year,Start Month,Total Deaths,No. Injured,Total Affected,No. Homeless,Total Damage ('000 US$),Magnitude,Latitude,Longitude
26,1982-0293-PER,Peru,AMERICAS,Latin America and the Caribbean,Earthquake,Ground movement,Geophysical,NaN,1982 AD,3.0,3.0,NaN,NaN,NaN,5000.0,6.1,-12.69,-76.065
138,2012-0219-GAB,Gabon,Africa,Sub-Saharan Africa,FLOOD,Riverine flood,Hydrological,NaN,Year 2012,6.0,1.0,NaN,77845.0,NaN,NaN,NaN,NaN,NaN
480,2021-0037-USA,U.S.A.,Americas,Northern America,Storm,Tornado,Meteorological,NaN,2021 AD,1.0,1.0,30.0,30.0,NaN,120000.0,NaN,NaN,NaN
506,2017-0483-IND,India,Asia,Southern Asia,Mass movement (wet),Landslide (wet),Hydrological,NaN,Year 2017,8.0,46.0,NaN,100.0,100.0,NaN,NaN,NaN,NaN
634,2003-0212-AZE,Azerbaijan,Asia,Western Asia,Flood,Riverine flood,Hydrological,NaN,Year 2003,4.0,NaN,NaN,31500.0,1500.0,55000.0,30870.0,NaN,NaN


In [17]:
# 3. Nettoyez la colonne : supprimez le texte et convertissez en numérique

df['Start Year'] = df['Start Year'].str.extract(r'(\d{4})')

# conversion en numérique
df['Start Year'] = pd.to_numeric(df['Start Year'], errors='coerce')

In [18]:
# 4. Vérifiez le résultat
df['Start Year'].dtype

dtype('int64')

le type est passé de O à int64

## Exercice 4 — Traitement des valeurs manquantes

In [24]:
# 1. Pour Region (peu de nulls) : supprimez les lignes avec null
region_null = df['Region'].isnull().sum()
print(region_null)

346


In [25]:
df = df.dropna(subset=['Region'])

In [26]:
# 2. Pour Event Name : remplacez les nulls par "Non nommé"
event_name_null = df['Event Name'].isnull().sum()
print(event_name_null)

13139


In [27]:
df['Event Name'] = df['Event Name'].fillna('Non nommé')

In [28]:
# 3. Pour Total Deaths : décidez d'une stratégie et appliquez-la
# On cherche la stratégie 
total_deaths_null = df['Total Deaths'].isnull().sum()
print(total_deaths_null)

4774


On a 4774 nulls sur Total Deaths.
On peut eliminé la situation "Peu de nulls, colonne critique". On peux garder la situation "Beaucoup de nulls, colonne non critique"

In [33]:
# on cherche la stratégie
# on regarde si Numérique, distribution asymétrique

print(df['Total Deaths'].dtype) # on rappel le type de la colonne
print(df['Total Deaths'].describe()) # on regarde les statistiques de la colonne
print(df['Total Deaths'].median())
print(df['Total Deaths'].mean())

float64
count    1.224000e+04
mean     2.773844e+03
std      6.650114e+04
min     -3.600000e+04
25%      5.000000e+00
50%      1.800000e+01
75%      6.100000e+01
max      3.700000e+06
Name: Total Deaths, dtype: float64
18.0
2773.843545751634


la distribution de Total Deaths est asymétrique. On peu imputer par la médiane.

In [34]:
# on cherche la stratégie
#on regarde si c'est catégoriel
df['Total Deaths'].nunique()

949

pas categoriel trop de valeurs différentes on perdrait de l'info si on impute par mode, et on ne peux pas mettre de "inconnu" ou "non renseigné" car on a des données chiffrées. 

Ca se joue entre 2 situations :
- Beaucoup de nulls, colonne non critique : on peut garder les nulls
- Numérique, distribution asymétrique : on peut imputer par la médiane

Il faut se placer dans le contexte métier pour prendre une decision :
- Pour Region (peu de nulls) : supprimez les lignes avec null
- Pour Event Name : remplacez les nulls par "Non nommé"
- Total Deaths : c’est l'une des variables principales d’analyse pour catastrophes mais il arrive que pour un evenement donné, on n’ait pas d’information sur le nombre de décès. De plus il y a des catastrophe qui peuvent engendrer 0 décès comme certains peuvent engendrer des milliers de décès. Donner le nombre total de mort est aussi fastidieux et peut être sujet à controverse surtout pour un gouvernement(Si on travail pour un gouv).

Décision prise : Situation "Beaucoup de nulls, colonne non critique". On garde les nulls.

In [35]:
# 4. Pour Magnitude : laissez les nulls (n'a pas de sens pour tous les types)
magnitude_null = df['Magnitude'].isnull().sum()
print(magnitude_null)

11929


In [37]:
df['Magnitude'].head(10)

0         NaN
1         NaN
2         NaN
3         NaN
4         6.8
5    365800.0
6         NaN
7         NaN
8         NaN
9         NaN
Name: Magnitude, dtype: float64

In [38]:
# 5. Vérifiez le résultat
df['Region'].isnull().sum()

np.int64(0)

In [39]:
df["Event Name"].isnull().sum()

np.int64(0)

In [41]:
# magnitude

In [40]:
df['Total Deaths'].isnull().sum()

np.int64(4774)

## Exercice 5 — Traitement des outliers

In [44]:
# 1. Identifiez les valeurs négatives dans Total Deaths
negative_total_deaths = df[df['Total Deaths'] < 0]
print("Valeurs négatives dans Total Deaths:\n", negative_total_deaths)

Valeurs négatives dans Total Deaths:
               DisNo.      Country    Region                        Subregion  \
33     2002-0849-GTM    Guatemala  Americas  Latin America and the Caribbean   
136    1961-0016-ETH     ethiopia    Africa               Sub-Saharan Africa   
293    1983-0437-JPN        Japan      Asia                     Eastern Asia   
607    1985-0009-IDN    Indonesia     Asia                South-eastern Asia   
754    1987-0330-USA   Etats-Unis  Americas                 Northern America   
...              ...          ...       ...                              ...   
16609  2023-0095-MDG   Madagascar    AFRICA               Sub-Saharan Africa   
16653  1999-0354-MMR      Myanmar      Asia               South-eastern Asia   
16790  1999-0728-PHL  Philippines      Asia               South-eastern Asia   
16851  1987-0344-CHN        China      Asia                     Eastern Asia   
16922  2018-0056-IDN    INDONESIA      Asia               South-eastern Asia   

 

un valeur negative dans Total Deaths n'a pas de sens.

In [45]:
df = df[df['Total Deaths'] >= 0]

In [47]:
# 2. Identifiez les valeurs aberrantes dans Magnitude (> 10)
outliers_magnitude = df[df['Magnitude'] > 10]
print("Valeurs aberrantes dans Magnitude (> 10):\n", outliers_magnitude)

Valeurs aberrantes dans Magnitude (> 10):
               DisNo.                           Country    Region  \
5      1997-0566-BOL  Bolivia (Plurinational State of)  Americas   
32     1996-0215-LAO  Lao People's Democratic Republic      Asia   
40     1995-0198-THA                          Thailand     Asia    
42     1996-0298-IND                             India      Asia   
43     2005-0756-ROU                           Romania    Europe   
...              ...                               ...       ...   
17455  1996-0029-GAB                             Gabon    africa   
17474  2007-0103-YEM                             Yemen     Asia    
17479  2008-0043-MDG                        MADAGASCAR    Africa   
17490  2003-0447-SOM                           Somalia    Africa   
17499  2022-0099-MDG                        Madagascar    Africa   

                             Subregion        Disaster Type  Disaster Subtype  \
5      Latin America and the Caribbean                flood

3. Décidez : supprimer ou corriger ?

- Supprimer les lignes avec des valeurs négatives dans Total Deaths car elles sont incohérentes et ne peuvent pas être corrigées de manière fiable.

- En revanche pour magnitude, on peut corriger les valeurs aberrantes en les limitant à une valeur maximale raisonnable (par exemple, 10) pour éviter de perdre des données potentiellement utiles.

In [ ]:
# 4. Appliquez le traitement
# Corriger les valeurs aberrantes de magnitude en les limitant à 10
# df['Magnitude'] = df['Magnitude'].clip(lower=0, upper=10)

df.loc[:, 'Magnitude'] = df['Magnitude'].clip(lower=0, upper=10)

In [50]:
# 5. Vérifiez le résultat
print("Statistiques de Total Deaths après nettoyage:\n", df['Total Deaths'].describe())
print("Statistiques de Magnitude après nettoyage:\n", df['Magnitude'].describe())


Statistiques de Total Deaths après nettoyage:
 count    1.217500e+04
mean     2.791854e+03
std      6.667720e+04
min     -0.000000e+00
25%      6.000000e+00
50%      1.800000e+01
75%      6.200000e+01
max      3.700000e+06
Name: Total Deaths, dtype: float64
Statistiques de Magnitude après nettoyage:
 count    3975.000000
mean        8.634299
std         2.353852
min         0.000000
25%         7.100000
50%        10.000000
75%        10.000000
max        10.000000
Name: Magnitude, dtype: float64


## Exercice 6 — Nettoyage du texte

In [52]:
# 1. Nettoyez Region : strip + title case

df.loc[:, 'Region'] = df['Region'].str.strip().str.title()

In [54]:
# 2. Vérifiez avec value_counts() — combien de catégories maintenant ?
df['Region'].value_counts()

Region
Asia        5479
Americas    2851
Africa      2169
Europe      1337
Oceania      339
Name: count, dtype: int64

In [ ]:
# 3. Nettoyez Disaster Type de la même manière
df.loc[:, 'Disaster Type'] = df['Disaster Type'].str.strip().str.title()

In [56]:
# 4. Nettoyez Country : strip + title case
df.loc[:, 'Country'] = df['Country'].str.strip().str.title()

In [63]:
# 5. Corrigez les variantes de pays (USA → United States, etc.)
df.loc[:, 'Country'] = df['Country'].replace({
    'Usa': 'United States',
    'United States Of America': 'United States'
})

In [64]:
# verif
print(df['Disaster Type'].unique()[:10])
print("-----")
print(df['Country'].unique()[:10])

['Flood' 'Storm' 'Epidemic' 'Earthquake' 'Mass Movement (Wet)'
 'Extreme Temperature' 'Wildfire' 'Mass Movement (Dry)'
 'Volcanic Activity' 'Drought']
-----
['Indonesia' 'Solomon Islands' 'Cambodia' 'Japon'
 'Bolivia (Plurinational State Of)' 'India' 'Honduras' 'Peru' 'Colombia'
 'Haiti']


In [68]:
unique_val_region2 = df["Region"].unique()
print("Valeurs uniques de la colonne 'Region' après nettoyage:\n", unique_val_region2)

Valeurs uniques de la colonne 'Region' après nettoyage:
 ['Asia' 'Oceania' 'Americas' 'Europe' 'Africa']


## EXERCICE 7 — Nouvelles colonnes

In [70]:
# 1. Créez une colonne Decennie à partir de Start Year
df.loc[:, 'Decennie'] = (df['Start Year'] // 10) * 10

In [72]:
# 2. Créez une colonne Has_Deaths (booléen : True si Total Deaths > 0)
df.loc[:, 'Has_Deaths'] = df['Total Deaths'] > 0

In [73]:
# 3. Vérifiez vos nouvelles colonnes
df[['Start Year', 'Decennie', 'Total Deaths', 'Has_Deaths']].head()

,Start Year,Decennie,Total Deaths,Has_Deaths
0,2020,2020,1.0,True
2,2015,2010,9.0,True
3,2007,2000,182.0,True
4,2008,2000,1.0,True
5,1997,1990,16.0,True


## EXERCICE 8 — Export

In [75]:
# Sélectionnez les colonnes utiles pour l'analyse

df_clean = df.copy()
df_clean 

,DisNo.,Country,Region,Subregion,Disaster Type,Disaster Subtype,Disaster Subgroup,Event Name,Start Year,Start Month,Total Deaths,No. Injured,Total Affected,No. Homeless,Total Damage ('000 US$),Magnitude,Latitude,Longitude,Decennie,Has_Deaths
0,2020-0121-IDN,Indonesia,Asia,South-eastern Asia,Flood,Flood (General),Hydrological,Non nommé,2020,3.0,1.0,NaN,56488.0,NaN,NaN,NaN,NaN,NaN,2020,True
2,2015-0281-SLB,Solomon Islands,Oceania,Melanesia,Storm,Tropical cyclone,Meteorological,Tropical cylone Raquel,2015,7.0,9.0,NaN,400.0,400.0,2000.0,NaN,NaN,NaN,2010,True
3,2007-0274-KHM,Cambodia,Asia,South-eastern Asia,Epidemic,Viral disease,Biological,Dengue,2007,7.0,182.0,NaN,17000.0,NaN,NaN,NaN,NaN,NaN,2000,True
4,2008-0275-JPN,Japon,Asia,Eastern Asia,Earthquake,Ground movement,Geophysical,Non nommé,2008,7.0,1.0,200.0,470.0,NaN,110000.0,6.8,39.802,141.464,2000,True
5,1997-0566-BOL,Bolivia (Plurinational State Of),Americas,Latin America and the Caribbean,Flood,Riverine flood,Hydrological,Non nommé,1997,3.0,16.0,NaN,NaN,NaN,35000.0,10.0,NaN,NaN,1990,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17504,1998-0024-USA,United States,Americas,Northern America,Storm,Tornado,Meteorological,Non nommé,1998,2.0,42.0,100.0,550.0,250.0,150000.0,NaN,NaN,NaN,1990,True
17506,2011-0303-DOM,Dominican Republic,Americas,Latin America and the Caribbean,Storm,Tropical cyclone,Meteorological,"Tropical storm ""Emily""",2011,8.0,3.0,NaN,7000.0,NaN,NaN,NaN,NaN,NaN,2010,True
17507,1995-0015-COL,Colombia,Americas,Latin America and the Caribbean,Earthquake,Ground movement,Geophysical,Non nommé,1995,1.0,7.0,35.0,2845.0,250.0,NaN,6.5,4.900,-73.100,1990,True
17508,1959-0031-HTI,Haiti,Americas,Latin America and the Caribbean,Flood,Flood (General),Hydrological,Non nommé,1959,4.0,50.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1950,True


In [82]:
# renomme les colonnes pour plus de clarté
df_clean = df_clean.rename(columns={
    'DisNo.': 'Disaster_id',
    'Country': 'Country',
    'Region': 'Region',
    'Subregion': 'Subregion',
    'Disaster Type': 'Disaster_Type',
    'Disaster Subtype': 'Disaster_Subtype',
    'Event Name': 'Event_Name',
    'Start Year': 'Start_Year',
    'Start Month': 'Start_Month',
    'Start Day': 'Start_Day',
    'Total Deaths': 'Total_Deaths',
    'Magnitude': 'Magnitude',
    'Decennie': 'Decennie',
    'Has_Deaths': 'Has_Deaths',
    'No. Injured': 'Nb_injured',
    'No. Homeless': 'Nb_homeless',
    'Total Affected': 'Total_Affected',
    'Total Damages (USD)': 'Total_Damages_USD',
    'latitude': 'Latitude',
    'longitude': 'Longitude'
})

In [83]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12175 entries, 0 to 17509
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Disaster_id              12175 non-null  object 
 1   Country                  12175 non-null  object 
 2   Region                   12175 non-null  object 
 3   Subregion                12175 non-null  object 
 4   Disaster_Type            12175 non-null  object 
 5   Disaster_Subtype         12175 non-null  object 
 6   Disaster Subgroup        12175 non-null  object 
 7   Event_Name               12175 non-null  object 
 8   Start_Year               12175 non-null  int64  
 9   Start_Month              11650 non-null  float64
 10  Total_Deaths             12175 non-null  float64
 11  Nb_injured               3824 non-null   float64
 12  Total_Affected           9075 non-null   float64
 13  Nb_homeless              1897 non-null   float64
 14  Total Damage ('000 US$)  42

comparaison avec le dataset initial :

```python   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   DisNo.                   17510 non-null  object 
 1   Country                  17510 non-null  object 
 2   Region                   17161 non-null  object 
 3   Subregion                17510 non-null  object 
 4   Disaster Type            17510 non-null  object 
 5   Disaster Subtype         17510 non-null  object 
 6   Disaster Subgroup        17510 non-null  object 
 7   Event Name               3975 non-null   object 
 8   Start Year               17510 non-null  object 
 9   Start Month              16599 non-null  float64
 10  Total Deaths             12593 non-null  float64
 11  No. Injured              4507 non-null   float64
 12  Total Affected           12951 non-null  float64
 13  No. Homeless             2521 non-null   float64
 14  Total Damage ('000 US$)  5630 non-null   float64
 15  Magnitude                5249 non-null   float64
 16  Latitude                 2821 non-null   float64
 17  Longitude                2821 non-null   float64
 ```

In [85]:
df_clean.head()

,Disaster_id,Country,Region,Subregion,Disaster_Type,Disaster_Subtype,Disaster Subgroup,Event_Name,Start_Year,Start_Month,Total_Deaths,Nb_injured,Total_Affected,Nb_homeless,Total Damage ('000 US$),Magnitude,Latitude,Longitude,Decennie,Has_Deaths
0,2020-0121-IDN,Indonesia,Asia,South-eastern Asia,Flood,Flood (General),Hydrological,Non nommé,2020,3.0,1.0,NaN,56488.0,NaN,NaN,NaN,NaN,NaN,2020,True
2,2015-0281-SLB,Solomon Islands,Oceania,Melanesia,Storm,Tropical cyclone,Meteorological,Tropical cylone Raquel,2015,7.0,9.0,NaN,400.0,400.0,2000.0,NaN,NaN,NaN,2010,True
3,2007-0274-KHM,Cambodia,Asia,South-eastern Asia,Epidemic,Viral disease,Biological,Dengue,2007,7.0,182.0,NaN,17000.0,NaN,NaN,NaN,NaN,NaN,2000,True
4,2008-0275-JPN,Japon,Asia,Eastern Asia,Earthquake,Ground movement,Geophysical,Non nommé,2008,7.0,1.0,200.0,470.0,NaN,110000.0,6.8,39.802,141.464,2000,True
5,1997-0566-BOL,Bolivia (Plurinational State Of),Americas,Latin America and the Caribbean,Flood,Riverine flood,Hydrological,Non nommé,1997,3.0,16.0,NaN,NaN,NaN,35000.0,10.0,NaN,NaN,1990,True


In [86]:
df_clean.to_csv("datas/catnat_clean.csv", index=False)

## Exercice 9 — Vérification dans Tableau

Lien du dashboard : 